In [1]:
import pandas as pd

df = pd.read_csv("/mnt/data/image_recognition/brown_forman_req/output/restaurant_brand_data_include_all_attributes_mentions_2.csv")

df.head()

df.columns



Index(['poi_code', 'restaurant_name', 'cost_for_two', 'absinthe_brand_name',
       'absinthe_item_price', 'beer_brand_name', 'beer_item_price',
       'brandy_brand_name', 'brandy_item_price', 'gin_brand_name',
       'gin_item_price', 'martini_brand_name', 'martini_item_price',
       'mezcal_brand_name', 'mezcal_item_price', 'mojito_brand_name',
       'mojito_item_price', 'old_fashioned_brand_name',
       'old_fashioned_item_price', 'other_brand_name', 'other_item_price',
       'other_cocktails_brand_name', 'other_cocktails_item_price',
       'picante_brand_name', 'picante_item_price', 'rum_brand_name',
       'rum_item_price', 'soju_brand_name', 'soju_item_price',
       'spirits_brand_name', 'spirits_item_price', 'tequila_brand_name',
       'tequila_item_price', 'vodka_brand_name', 'vodka_item_price',
       'whisky_brand_name', 'whisky_item_price', 'wine_brand_name',
       'wine_item_price', ' mentions', 'location_status', 'reviews',
       'offerings__serves_cocktails', 'a

In [2]:
import pandas as pd
import re
import ast
 
# ═══════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════
 
def is_true(row, col):
    """Safely handle TRUE/FALSE strings from CSV"""
    return str(row.get(col, "")).strip().upper() == "TRUE"
 
def parse_cost(val):
    """'₹3,500 for two' → 3500"""
    if pd.isna(val): return None
    match = re.search(r'\d[\d,]+', str(val))
    return int(match.group().replace(",", "")) if match else None
 

def safe_num(val, cast=float, default=0):
    """Safely cast a value that may be NaN, None, or empty string."""
    try:
        return cast(val) if pd.notna(val) and str(val).strip() != "" else default
    except (ValueError, TypeError):
        return default


def cost_to_score(cost):
    """
    Mutually exclusive price tiers.
    Based on Mumbai market: ₹600 street food → ₹5000+ fine dining.
    """
    if cost is None:  return None
    if cost > 4000:   return 8
    if cost > 2500:   return 6
    if cost > 1200:   return 3
    if cost > 600:    return 1
    return 0
 
# ═══════════════════════════════════════════════════════════
# PROXY 1 — Infer spend level from drink/brand item prices
# ═══════════════════════════════════════════════════════════
 
DRINK_PRICE_COLS = [
    'beer_item_price', 'gin_item_price', 'whisky_item_price',
    'vodka_item_price', 'rum_item_price', 'tequila_item_price',
    'martini_item_price', 'wine_item_price', 'mojito_item_price',
    'other_item_price', 'spirits_item_price', 'other_cocktails_item_price',
    'mezcal_item_price', 'soju_item_price', 'picante_item_price',
    'absinthe_item_price', 'brandy_item_price', 'old_fashioned_item_price'
]
 
def infer_cost_from_drinks(row):
    """
    Median drink price × 6 ≈ implied cost for two.
    Rationale: alcohol typically 15–20% of total bill.
    A cocktail at ₹800+ → premium venue; ₹250 → casual bar.
    Receives -1 score haircut downstream (inferred, not observed).
    """
    prices = []
    for col in DRINK_PRICE_COLS:
        val = row.get(col)
        if pd.notna(val) and str(val).strip().lower() not in ("", "nan", "other"):
            try:
                prices.append(float(str(val).replace(",", "")))
            except ValueError:
                pass
    if not prices:
        return None
    median_drink = sorted(prices)[len(prices) // 2]
    return median_drink * 6
 
# ═══════════════════════════════════════════════════════════
# PROXY 2 — Review text mining
# Three signal types + one negative/penalty type
# ═══════════════════════════════════════════════════════════
 
# (+) Price keywords: explicit spend mentions
PRICE_KW = [
    '₹2,000', '₹3,000', '₹4,000', '₹5,000',
    '₹2000',  '₹3000',  '₹4000',  '₹5000',
    'pricey', 'premium', 'expensive', 'steep', 'splurge',
    'worth every penny', 'high end', 'high-end', 'not cheap',
    'costs a lot', 'on the pricier side', 'pocket pinch'
]
 
# (+) Experience/quality keywords
EXPERIENCE_KW = [
    'fine dining', 'upscale', 'luxur', 'exclusive', 'world class',
    'top notch', 'exceptional', 'impeccable', 'michelin',
    'sophisticated', 'elevated', 'white glove', 'curated menu',
    'chef', 'sommelier', 'tasting menu'
]
 
# (+) Crowd/vibe keywords
VIBE_KW = [
    'rooftop', 'intimate', 'romantic', 'buzzing', 'lively',
    'date night', 'special occasion', 'anniversary', 'celebration',
    'chic', 'trendy', 'vibrant', 'ambience', 'atmosphere',
    'stunning view', 'beautiful decor', 'instagrammable'
]
 
# (−) Negative price shock: bad value signals → subtract score
NEGATIVE_KW = [
    'overpriced', 'not worth', 'too expensive', 'rip off', 'ripoff',
    'over priced', 'not worth the price', 'not worth the hype',
    'daylight robbery', 'exorbitant', 'highway robbery',
    'waste of money', 'poor value', 'not worth it'
]
 
def mine_reviews(row):
    """
    Parse the raw reviews list, scan for all four keyword buckets.
    Returns: (price_hits, experience_hits, vibe_hits, negative_hits)
    """
    raw = row.get("reviews", "[]")
    price_hits = experience_hits = vibe_hits = negative_hits = 0
    try:
        reviews = ast.literal_eval(raw) if isinstance(raw, str) else (raw or [])
        full_text = " ".join(str(r) for r in reviews).lower()
 
        price_hits      = sum(1 for kw in PRICE_KW      if kw.lower() in full_text)
        experience_hits = sum(1 for kw in EXPERIENCE_KW if kw.lower() in full_text)
        vibe_hits       = sum(1 for kw in VIBE_KW        if kw.lower() in full_text)
        negative_hits   = sum(1 for kw in NEGATIVE_KW    if kw.lower() in full_text)
    except Exception:
        pass
    return price_hits, experience_hits, vibe_hits, negative_hits
 
# ═══════════════════════════════════════════════════════════
# MASTER CLASSIFIER
# ═══════════════════════════════════════════════════════════
 
def classify_restaurant(row):
    score = 0
    price_source = "direct"
 
    # ── LAYER 1: Direct cost_for_two ──────────────────────
    cost = parse_cost(row.get("cost_for_two"))
    price_score = cost_to_score(cost)
 
    # ── LAYER 2: Fallback → infer from drink item prices ──
    if price_score is None:
        implied_cost = infer_cost_from_drinks(row)
        price_score  = cost_to_score(implied_cost)
        price_source = "drinks_inferred"
        if price_score is not None:
            price_score = max(0, price_score - 1)   # haircut: inferred data
 
    # ── LAYER 3: Fallback → bootstrap from reviews alone ──
    price_hits, experience_hits, vibe_hits, negative_hits = mine_reviews(row)
 
    if price_score is None:
        price_score  = min(price_hits, 2) + min(experience_hits, 2)
        price_source = "reviews_inferred"
 
    score += (price_score or 0)
 
    # ── Review signals (additive regardless of price source)
    score += min(experience_hits, 2)   # "fine dining", "upscale" → +2 max
    score += min(vibe_hits, 2)         # "rooftop", "romantic"   → +2 max
    score += min(price_hits, 1)        # price mentions          → +1 max (softer)
    score -= min(negative_hits * 2, 4) # bad value penalty       → -2 per hit, -4 max
 
    # ── Structural flags ──────────────────────────────────
    flag_map = {
        "atmosphere__feels_upscale":                    2,
        "atmosphere__feels_romantic":                   1,
        "planning__requires_reservations":              1,
        "planning__recommends_reservations_dinner":     1,
        "parking__has_parking_valet":                   1,
        "offerings__has_private_dining_room":           2,
        "highlights__has_live_music":                   1,
        "highlights__has_seating_rooftop":              1,
        "offerings__serves_cocktails":                  1,
        "offerings__serves_wine":                       1,
        "atmosphere__feels_hip":                        1,
        "offerings__serves_happy_hour_drinks_x":        1,
    }
    for col, pts in flag_map.items():
        if is_true(row, col):
            score += pts
 
    # ── Drink menu depth (breadth = sophistication proxy) ─
    brand_cols = [c for c in row.index if c.endswith("_brand_name")]
    drink_depth = sum(
        1 for c in brand_cols
        if pd.notna(row.get(c))
        and str(row.get(c)).strip().lower() not in ("", "nan", "other")
    )
    if drink_depth > 6:   score += 3
    elif drink_depth > 3: score += 2
    elif drink_depth > 0: score += 1
 
    # ── Ratings ───────────────────────────────────────────
    rating = safe_num(row.get("ratings"))
    if rating >= 4.5:    score += 2
    elif rating >= 4.0:  score += 1
    elif rating < 3.5:   score -= 1   # poor rating = soft penalty
 
    # ── Review volume (popularity signal) ─────────────────
    reviews_count = safe_num(row.get("reviews_count"))
    if reviews_count > 3000:   score += 2
    elif reviews_count > 1000: score += 1
 
    # ── Segment thresholds ────────────────────────────────
    # Calibrated against your 4 sample restaurants:
    # O Pedro (BKC): expect Luxury    ~score 18+
    # Bandra Born:   expect Luxury    ~score 16+
    # La Loca Maria: expect Premium   ~score 13+
    # Mizu:          expect Premium   ~score 12+
    # Veronica's:    expect Budget    ~score 4
    if score >= 16:   segment = "Luxury"
    elif score >= 10: segment = "Premium"
    elif score >= 5:  segment = "Mid"
    else:             segment = "Budget"
 
    return pd.Series({
        "restaurant_segment":    segment,
        "segment_score":         score,
        "segment_price_source":  price_source,
    })

In [3]:
df[["restaurant_segment", "segment_score", "segment_price_source"]] = df.apply(classify_restaurant, axis=1)

In [4]:
df.head(5)

,poi_code,restaurant_name,cost_for_two,absinthe_brand_name,absinthe_item_price,beer_brand_name,beer_item_price,brandy_brand_name,brandy_item_price,gin_brand_name,...,atmosphere__feels_casual,atmosphere__feels_quiet,offerings__serves_wine,amenities__has_bar_onsite,planning__accepts_reservations,offerings__serves_happy_hour_drinks,offerings__serves_liquor,restaurant_segment,segment_score,segment_price_source
0,0x115aedf5dd67b353:0x7610f3ee573e0cda,Persian Darbar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,False,True,False,False,Luxury,17,reviews_inferred
1,0x3a35e54988ce8c69:0x9c0d1b573a855ba0,Hotel Karl Residency,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,Premium,10,reviews_inferred
2,0x3bae16773e7dc413:0x46098eee49dc03f2,Symphony restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,False,True,True,True,True,True,Premium,14,reviews_inferred
3,0x3bc123ac2a55dd51:0x8abbedb48e36dbe1,TEAMAX CAFFE,₹350 for two,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,False,False,False,Budget,4,direct
4,0x3bc2b9ee2d315a49:0x780fe1c1dd51e23,Blue Tokai Coffee Roasters | Phoenix Marketcit...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,False,False,False,False,False,False,Mid,9,reviews_inferred


In [5]:
df.to_csv("/mnt/data/image_recognition/brown_forman_req/output/classified_restaurant_brand_data_include_all_attributes_mentions.csv", index=False)

In [6]:
# Audit: distribution check
print("\n── Segment Distribution ──")
print(df["restaurant_segment"].value_counts())

print("\n── Price Source Breakdown ──")
print(df["segment_price_source"].value_counts())

print("\n── Score Stats ──")
print(df["segment_score"].describe())

# Spot-check inferred restaurants (lowest confidence)
inferred = df[df["segment_price_source"] != "direct"]
print(f"\n── {len(inferred)} restaurants classified via fallback ──")
print(inferred[["restaurant_name", "segment_score",
                    "segment_price_source", "restaurant_segment"]].to_string())


── Segment Distribution ──
restaurant_segment
Budget     1702
Mid        1611
Premium     873
Luxury      540
Name: count, dtype: int64

── Price Source Breakdown ──
segment_price_source
reviews_inferred    2599
direct              1799
drinks_inferred      328
Name: count, dtype: int64

── Score Stats ──
count    4726.000000
mean        7.621456
std         5.698376
min        -4.000000
25%         3.000000
50%         6.000000
75%        11.000000
max        30.000000
Name: segment_score, dtype: float64

── 2927 restaurants classified via fallback ──
                                                                                                                    restaurant_name  segment_score segment_price_source restaurant_segment
0                                                                                                                    Persian Darbar             17     reviews_inferred             Luxury
1                                                                 